In [ ]:
from typing import Any
import matplotlib.pyplot as plt
import torch
import shapely
import numpy as np
import geopandas as gpd
import torchgeo.models
from torchgeo.datasets import Sentinel, stack_samples
from torchgeo.samplers import GridGeoSampler
from tqdm import tqdm

torch.multiprocessing.set_start_method('fork', force=True)


class Sentinel2AnnualMosaic(Sentinel):
    # example filename 16SEA_B9_2024-01-01_2025-01-01.tif
    filename_glob = "*.tif"
    filename_regex = r"(?P<tile>\d{2}[A-Z]{3})_(?P<band>B\d{1,2}A?)_(?P<start>\d{4}-\d{2}-\d{2})_(?P<stop>\d{4}-\d{2}-\d{2})\.tif"
    separate_files = True
    is_image = True
    date_format = '%Y-%m-%d'
    all_bands = (
        'B1',
        'B2',
        'B3',
        'B4',
        'B5',
        'B6',
        'B7',
        'B8',
        'B8A',
        'B9',
        'B11',
        'B12',
    )
    rgb_bands = ('B4', 'B3', 'B2')

    def plot(
        self,
        sample: dict[str, Any],
        show_titles: bool = True,
        suptitle: str | None = None,
    ) -> plt.Figure:
        rgb_indices = []
        for band in self.rgb_bands:
            if band in self.bands:
                rgb_indices.append(self.bands.index(band))
            else:
                raise ValueError(f"Band {band} not found in dataset bands: {self.bands}")

        image = sample['image'][rgb_indices].permute(1, 2, 0)
        image = (image / 3000).clip(0, 1)
        fig, ax = plt.subplots(1, 1, figsize=(4, 4))
        ax.imshow(image)
        ax.axis('off')

        if show_titles:
            ax.set_title('Image')

        if suptitle is not None:
            plt.suptitle(suptitle)

        return fig

root = "16SEA"
dataset = Sentinel2AnnualMosaic(paths=root, res=10, cache=False)
sampler = GridGeoSampler(dataset, size=256, stride=128)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, num_workers=12, sampler=sampler, collate_fn=stack_samples)
print(len(dataset), len(sampler))

device = "mps"
dtype = torch.float16

12 43056


In [ ]:
model = torchgeo.models.RCF(in_channels=12, features=16, kernel_size=3, mode="gaussian")

embeddings, geometries = [], []
with torch.amp.autocast(device_type=device, dtype=dtype):
    model.eval()
    model = model
    for batch in tqdm(dataloader, total=len(dataloader)):
        image = batch["image"]
        emb = model(image).cpu().numpy()
        embeddings.append(emb)
        for bound in batch["bounds"]:
            x, y, _ = batch["bounds"][0]
            centroid = ((x.start + x.stop) / 2, (y.start + y.stop) / 2)
            geom = shapely.geometry.Point(*centroid)
            geometries.append(geom)

embeddings = np.concatenate(embeddings, axis=0)

gdf = gpd.GeoDataFrame(
    data={"embedding": [emb.tolist() for emb in embeddings]},
    geometry=geometries,
    crs=dataset.crs,
)
gdf.to_crs(epsg=4326, inplace=True)
gdf.to_parquet("16SEA.parquet")

 64%|██████▎   | 3425/5382 [12:53<05:43,  5.70it/s]